# 03 — Level 1 TF-IDF experiments (RQ4)
Primary full multi-label cohort, fixed model/vary representation, plus logistic baseline and single-label sensitivity. Reuses the frozen leakage-aware split and resumes from saved result JSON files.


In [ ]:
from pathlib import Path
import json, subprocess, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass
REPO=Path('/content/research_software_classification_attributes')
if not REPO.exists():
    subprocess.run(['git','clone','-b','data-finalization','https://github.com/kuefmz/research_software_classification_attributes.git',str(REPO)],check=True)
sys.path.insert(0,str(REPO))
ROOT=Path('/content/drive/MyDrive/phd_research_software')
DATA=ROOT/'data/frozen/level1_high_level.jsonl'
SPLIT=ROOT/'splits/level1_high_level_group_aware_seed42.csv'
OUT=ROOT/'results/level1'; OUT.mkdir(parents=True,exist_ok=True)


In [ ]:
from src.experiments.classification import select_records,run_multilabel_tfidf,run_singlelabel_tfidf
representations=['paper_title','paper_abstract','paper_title_abstract','software_name','software_description','repository_title','readme','somef_description','PUBLICATION','SOFTWARE_REPOSITORY','ALL_AVAILABLE']
sources=['papers_with_code','bio.tools']
for source in sources:
    parts=select_records(DATA,SPLIT,source=source)
    for rep in representations:
        for model in ['tfidf_linearsvc','tfidf_logistic_regression']:
            dest=OUT/f'{source.replace(".","_")}__multilabel__{rep}__{model}.json'
            if dest.exists():
                print('skip',dest.name); continue
            try:
                result=run_multilabel_tfidf(parts=parts,representation=rep,model_name=model,seed=42)
                dest.write_text(json.dumps(result,indent=2)+'\n')
                print('saved',dest.name,result['test_metrics'])
            except ValueError as e:
                print('not usable',source,rep,model,e)


In [ ]:
for source in sources:
    parts=select_records(DATA,SPLIT,source=source,single_label_only=True)
    for rep in ['PUBLICATION','SOFTWARE_REPOSITORY','ALL_AVAILABLE']:
        for model in ['tfidf_linearsvc','tfidf_logistic_regression']:
            dest=OUT/f'{source.replace(".","_")}__singlelabel_sensitivity__{rep}__{model}.json'
            if dest.exists(): continue
            result=run_singlelabel_tfidf(parts=parts,representation=rep,model_name=model,seed=42)
            dest.write_text(json.dumps(result,indent=2)+'\n')
            print('saved',dest.name,result['test_metrics'])
